In [1]:
import numpy as np
import pandas as pd

from sklearn.metrics import brier_score_loss, average_precision_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
import xgboost as xgb

from cdc_ml.config import POLLS_PROCESSED,PREFERENCE_PROCESSED,CUSTOMER_CLASS_PROCESSED
from cdc_ml.config import MODELS_DIR, PROCESSED_DATA_DIR, STAGE_1_PROCESSED
from cdc_ml.features.build_features import drop_meta_high_card_cols
from cdc_ml.features.build_features import assign_class_type,drop_meta_high_card_cols
from cdc_ml.modeling.baseline import *

2026-06-03 21:24:04.708 | INFO     | cdc_ml.config:<module>:12 - PROJ_ROOT path is: C:\Users\zhiju\Desktop\cdc_ml


In [2]:
df = pd.read_parquet(POLLS_PROCESSED)
df_pref = pd.read_parquet(PREFERENCE_PROCESSED)
df_class = pd.read_parquet(CUSTOMER_CLASS_PROCESSED)

In [3]:
df= assign_class_type(df,df_class)
df_pref_melt = df_pref.melt(
    id_vars=["id","username","day_of_week","date"],
    value_vars=["t_0830","t_1020","t_1245","t_1435","t_1625","t_1850","t_2040"],
    var_name="timeslot",
    value_name="selected"

)
# temp = df_pref_melt.groupby(["id","timeslot"])["selected"].sum()
# print(temp)
pref_days_count = df_pref.groupby("id").size().reset_index(name="pref_days_count")
#print(total_pref_occurance)


pref_dow = df_pref.groupby(["id","day_of_week"]).size().reset_index(name="pref_dow_count")
#print(pref_dow)


df_pref_melt = df_pref_melt.loc[df_pref_melt["selected"]==1].drop_duplicates(subset=["id","day_of_week","timeslot"]).sort_values(by="id")
#print(df_pref_melt.head(5))
coverage = df_pref_melt.groupby(["id"]).size().reset_index(name="pref_coverage")
#print(width)

pref_unique_day = df_pref.groupby(["id"])["day_of_week"].nunique().reset_index(name="pref_unique_day")
#print(pref_unique_day)


pref_unique_timeslot = df_pref_melt.groupby(["id"])["timeslot"].nunique().reset_index(name="pref_unique_timeslot")
print(pref_unique_timeslot)





df = df.merge(coverage,on='id',how="left")
df = df.merge(pref_days_count,on="id",how="left")
df = df.merge(pref_unique_day,on="id",how="left")
df = df.merge(pref_unique_timeslot,on="id",how="left")

pref_dow_wide = (
    pref_dow
    .pivot(index="id", columns="day_of_week", values="pref_dow_count")
    .reindex(columns=range(7))        # force all 7 dows even if some never appear
    .fillna(0)                        # no row for a day => 0 prefs that day
    .astype("int64")
    .add_prefix("pref_dow_count_")    # pref_dow_count_0 ... pref_dow_count_6
    .rename_axis(columns=None)
    .reset_index()
)

n_before = len(df)
df = df.merge(pref_dow_wide, on="id", how="left")
assert len(df) == n_before, "merge fanned out — pref_dow_wide isn't unique per id"
df.head()

      id  pref_unique_timeslot
0      0                     7
1      1                     7
2      2                     7
3      3                     7
4      4                     5
..   ...                   ...
115  117                     7
116  118                     7
117  119                     3
118  120                     2
119  121                     7

[120 rows x 2 columns]


,id,username,cycle_start,cycle_end,polling_at,has_booking,cycle_start_month,cycle_start_day,cycle_start_dow,cycle_start_hour,...,pref_days_count,pref_unique_day,pref_unique_timeslot,pref_dow_count_0,pref_dow_count_1,pref_dow_count_2,pref_dow_count_3,pref_dow_count_4,pref_dow_count_5,pref_dow_count_6
0,0,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 09:00:00+08:00,False,8,19,1,9,...,13.0,7.0,7.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0
1,0,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 10:00:00+08:00,False,8,19,1,9,...,13.0,7.0,7.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0
2,0,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 11:00:00+08:00,False,8,19,1,9,...,13.0,7.0,7.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0
3,0,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 12:00:00+08:00,False,8,19,1,9,...,13.0,7.0,7.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0
4,0,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 13:00:00+08:00,False,8,19,1,9,...,13.0,7.0,7.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0


In [4]:
baseline = ["polling_hour","polling_dow"]
pref_coverage=["pref_coverage"]
pref_days_count=["pref_days_count"]
pref_unique_day=["pref_unique_day"]
pref_unique_timeslot=["pref_unique_timeslot"]
pref_dow=["pref_dow_count_0","pref_dow_count_1","pref_dow_count_2","pref_dow_count_3","pref_dow_count_4","pref_dow_count_5","pref_dow_count_6"]
class_features = ["is_one_team","class_type"]
test_list = {
    "baseline":                    baseline,
    "baseline+pref_unique_day":                    baseline+pref_unique_day,
    "baseline+pref_unique_timeslot":                    baseline+pref_unique_timeslot,
    "baseline+pref_unique_day+pref_unique_timeslot":                    baseline+pref_unique_day+pref_unique_timeslot,
    "baseline+pref_days_count":baseline+pref_days_count,
    "baseline+pref_coverage":                    baseline+pref_coverage,
    "baseline+pref_coverage+pref_days_count":baseline+pref_coverage+pref_days_count,
    "baseline+pref_unique_day+pref_unique_timeslot+pref_days_count":                    baseline+pref_unique_day+pref_unique_timeslot+pref_days_count,
    "baseline+pref_unique_day+pref_unique_timeslot+pref_days_count+pref_coverage":                    baseline+pref_unique_day+pref_unique_timeslot+pref_days_count+pref_coverage,
    "baseline+pref_unique_day+pref_unique_timeslot+pref_days_count+pref_coverage+class_feature":                    baseline+pref_unique_day+pref_unique_timeslot+pref_days_count+pref_coverage+class_features,
}

In [5]:
def train(
    df: pd.DataFrame,
    keep_features: list,
    param_grid: list | None = None,
    outer_splits: int = 5,
    inner_splits: int = 3,
    seed: int = 42,
):
    # assumes np, pd, xgb, RandomForestClassifier, StratifiedGroupKFold,
    # permutation_importance, brier_score_loss, average_precision_score and the
    # baseline fns are already imported/defined.
    y = df["has_booking"].to_numpy()
    X = df[keep_features]
    groups = df["username"].to_numpy()
    n = len(y)
    base = y.mean()

    # XGB grid searched in the INNER loop, selected on PR-AUC.
    # Replaces the old fixed lr/m_depth/min_child/reg_lamb args; centred on them.
    if param_grid is None:
        param_grid = [
            {"learning_rate": lr, "max_depth": d, "min_child_weight": mc, "reg_lambda": rl}
            for lr in (0.03, 0.1)
            for d in (2, 3,4)
            for mc in (10, 50)
            for rl in (1.0, 10.0)
        ]

    def _make_xgb(params):
        return xgb.XGBClassifier(
            n_estimators=500,
            objective="binary:logistic",
            subsample=0.8,
            colsample_bytree=0.8,
            tree_method="hist",
            random_state=seed,
            n_jobs=-1,
            **params,
        )

    BASELINES = {
        "const": lambda Xt, yt, Xv: baseline_const(Xt, yt),
        "marg_dow": marg_dow_baseline,
        "marg_hour": marg_hour_baseline,
        "add": additive_lut_logit,
        "joint": joint_lut_hier,
    }
    model_names = (*BASELINES, "xgb", "rf")

    oof = {name: np.full(n, np.nan) for name in model_names}
    fold_brier = {name: [] for name in model_names}
    fold_pr = {name: [] for name in model_names}
    tr_brier_list, tr_pr_list = [], []
    models, fold_params, importances, importances_std = [], [], [], []

    whales_pt_mask = df["username"].isin(["anmol", "jy", "mya"]).to_numpy()
    whales_pc_mask = df["username"].isin(["kim", "jy", "flower"]).to_numpy()

    outer = StratifiedGroupKFold(n_splits=outer_splits, shuffle=True, random_state=120)
    for fold, (tr, va) in enumerate(outer.split(X, y=y, groups=groups)):
        X_tr, y_tr, g_tr = X.iloc[tr], y[tr], groups[tr]
        X_va, y_va = X.iloc[va], y[va]

        # --- inner CV: pick XGB hyperparameters on outer-train only ---
        inner = StratifiedGroupKFold(n_splits=inner_splits, shuffle=True, random_state=121)
        best_score, best_params = -np.inf, None
        for params in param_grid:
            scores = []
            for itr, iva in inner.split(X_tr, y=y_tr, groups=g_tr):
                m = _make_xgb(params)
                m.fit(X_tr.iloc[itr], y_tr[itr])
                p = m.predict_proba(X_tr.iloc[iva])[:, 1]
                scores.append(average_precision_score(y_tr[iva], p))
            s = float(np.mean(scores))
            if s > best_score:
                best_score, best_params = s, params
        fold_params.append(best_params)

        # --- refit XGB on full outer-train with selected params, predict outer-test ---
        model = _make_xgb(best_params)
        model.fit(X_tr, y_tr)
        models.append(model)
        oof["xgb"][va] = model.predict_proba(X_va)[:, 1]
        tr_pred = model.predict_proba(X_tr)[:, 1]

        # --- RandomForest (untuned sanity baseline) ---
        rf = RandomForestClassifier(
            n_estimators=500,
            max_depth=6,
            min_samples_leaf=50,
            max_features=None,  # only 2 features — let every split see both; diversity from bagging
            n_jobs=-1,
            random_state=seed,
        )
        rf.fit(X_tr, y_tr)
        oof["rf"][va] = rf.predict_proba(X_va)[:, 1]

        # --- permutation importance on outer-test, selected model ---
        r = permutation_importance(
            model, X_va, y_va,
            n_repeats=10, random_state=seed, n_jobs=1,
            scoring="average_precision",
        )
        importances.append(r.importances_mean)
        importances_std.append(r.importances_std)

        # --- baselines (untuned) ---
        for name, fn in BASELINES.items():
            oof[name][va] = fn(X_tr, y_tr, X_va)

        # --- per-fold scores ---
        fb, fp = {}, {}
        for name in model_names:
            fb[name] = brier_score_loss(y_va, oof[name][va])
            fp[name] = average_precision_score(y_va, oof[name][va])
            fold_brier[name].append(fb[name])
            fold_pr[name].append(fp[name])
        tr_brier = brier_score_loss(y_tr, tr_pred)
        tr_pr = average_precision_score(y_tr, tr_pred)
        tr_brier_list.append(tr_brier)
        tr_pr_list.append(tr_pr)

        print(
            f"fold {fold}: train n={len(tr):>6} val n={len(va):>6}"
            f"  train_pos={y_tr.mean():.3f}  val_pos={y_va.mean():.3f}\n"
            f"  picked    {best_params}  (inner pr={best_score:.4f})\n"
            f"  marg_dow  brier={fb['marg_dow']:.4f}  pr={fp['marg_dow']:.4f}\n"
            f"  marg_hour brier={fb['marg_hour']:.4f}  pr={fp['marg_hour']:.4f}\n"
            f"  add       brier={fb['add']:.4f}  pr={fp['add']:.4f}\n"
            f"  joint     brier={fb['joint']:.4f}  pr={fp['joint']:.4f}\n"
            f"  rf        brier={fb['rf']:.4f}  pr={fp['rf']:.4f}\n"
            f"  xgb (val) brier={fb['xgb']:.4f}  pr={fp['xgb']:.4f}\n"
            f"  xgb (tr)  brier={tr_brier:.4f}  pr={tr_pr:.4f}\n"
        )

    # --- segment base rates ---
    whale_pt_base = y[whales_pt_mask].mean()
    non_whale_pt_base = y[~whales_pt_mask].mean()
    whale_pc_base = y[whales_pc_mask].mean()
    non_whale_pc_base = y[~whales_pc_mask].mean()

    def _row(name):
        b = brier_score_loss(y, oof[name])
        p = average_precision_score(y, oof[name])
        bl, pl = fold_brier[name], fold_pr[name]
        return (
            f"{name:<9} brier={b:.4f} ({np.mean(bl):.4f}±{np.std(bl):.4f}) lift={b / base}   "
            f"pr_auc={p:.4f} ({np.mean(pl):.4f}±{np.std(pl):.4f}) lift={p / base}"
        )

    def _segment(arr):
        # brier uses pc whales (calibration set), pr uses pt whales (enough positives)
        wb = brier_score_loss(y[whales_pc_mask], arr[whales_pc_mask])
        nwb = brier_score_loss(y[~whales_pc_mask], arr[~whales_pc_mask])
        wp = average_precision_score(y[whales_pt_mask], arr[whales_pt_mask])
        nwp = average_precision_score(y[~whales_pt_mask], arr[~whales_pt_mask])
        return (
            f"          whales     brier={wb} lift={wb / whale_pc_base}\n"
            f"          non-whales brier={nwb} lift={nwb / non_whale_pc_base}\n"
            f"          whales     pr={wp} lift={wp / whale_pt_base}\n"
            f"          non-whales pr={nwp} lift={nwp / non_whale_pt_base}"
        )

    avg_df = pd.DataFrame(
        {
            "feature": X.columns,
            "mean": np.mean(importances, axis=0),
            "std": np.mean(importances_std, axis=0),
        }
    ).sort_values("mean", ascending=False)

    print(
        f"\nOOF  base_rate={base:.4f}  n={len(y)}/{n}      "
        f"format: pooled (mean±std over folds)\n"
        f"{_row('const')}\n"
        f"{_row('marg_dow')}\n"
        f"{_row('marg_hour')}\n"
        f"{_row('add')}\n"
        f"{_row('joint')}\n"
        f"{_segment(oof['joint'])}\n"
        f"{_row('rf')}\n"
        f"{_segment(oof['rf'])}\n"
        f"{_row('xgb')}\n"
        f"{_segment(oof['xgb'])}\n"
        f"xgb_tr    brier=  ---   ({np.mean(tr_brier_list):.4f}±{np.std(tr_brier_list):.4f})   "
        f"pr_auc=  ---   ({np.mean(tr_pr_list):.4f}±{np.std(tr_pr_list):.4f})\n"
        f"selected params per fold: {fold_params}\n"
        f"Average importances\n{avg_df}"
    )

    return oof["xgb"], oof["add"], models, whales_pc_mask

In [6]:
df.head()

,id,username,cycle_start,cycle_end,polling_at,has_booking,cycle_start_month,cycle_start_day,cycle_start_dow,cycle_start_hour,...,pref_days_count,pref_unique_day,pref_unique_timeslot,pref_dow_count_0,pref_dow_count_1,pref_dow_count_2,pref_dow_count_3,pref_dow_count_4,pref_dow_count_5,pref_dow_count_6
0,0,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 09:00:00+08:00,False,8,19,1,9,...,13.0,7.0,7.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0
1,0,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 10:00:00+08:00,False,8,19,1,9,...,13.0,7.0,7.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0
2,0,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 11:00:00+08:00,False,8,19,1,9,...,13.0,7.0,7.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0
3,0,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 12:00:00+08:00,False,8,19,1,9,...,13.0,7.0,7.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0
4,0,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 13:00:00+08:00,False,8,19,1,9,...,13.0,7.0,7.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0


In [8]:
#chosen = baseline+pref_unique_day+pref_unique_timeslot+pref_days_count+pref_coverage
chosen = baseline
oof_xgb = train(df,chosen)

fold 0: train n= 25874 val n=  6618  train_pos=0.012  val_pos=0.011
  picked    {'learning_rate': 0.03, 'max_depth': 2, 'min_child_weight': 10, 'reg_lambda': 10.0}  (inner pr=0.0305)
  marg_dow  brier=0.0109  pr=0.0137
  marg_hour brier=0.0108  pr=0.0270
  add       brier=0.0108  pr=0.0310
  joint     brier=0.0109  pr=0.0255
  rf        brier=0.0108  pr=0.0275
  xgb (val) brier=0.0108  pr=0.0313
  xgb (tr)  brier=0.0122  pr=0.0330

fold 1: train n= 25619 val n=  6873  train_pos=0.013  val_pos=0.010
  picked    {'learning_rate': 0.03, 'max_depth': 3, 'min_child_weight': 10, 'reg_lambda': 1.0}  (inner pr=0.0329)
  marg_dow  brier=0.0098  pr=0.0136
  marg_hour brier=0.0097  pr=0.0212
  add       brier=0.0097  pr=0.0288
  joint     brier=0.0097  pr=0.0289
  rf        brier=0.0097  pr=0.0294
  xgb (val) brier=0.0097  pr=0.0291
  xgb (tr)  brier=0.0125  pr=0.0343

fold 2: train n= 25835 val n=  6657  train_pos=0.013  val_pos=0.011
  picked    {'learning_rate': 0.03, 'max_depth': 3, 'min_chil